# Interactions and scaling of regression inputs

## Data

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

df = pd.read_csv("../../data/polish-jvs.csv", dtype={"id": np.int64, "woj":str, "public":str,"size": str, "nace_division": str, "nace": str})
nace_counts = df.groupby("nace_division").size().reset_index(name="n_firms")
df = df.merge(nace_counts, on="nace_division")
df["n_firms"] = df["n_firms"] / 1000  ## in thousands for readable coefficients
print(df[["nace_division", "size", "n_firms", "vacancies"]].head())

## Model without interaction (parallel lines)

In [ ]:
model_par = smf.ols("vacancies ~ size + n_firms", data = df).fit()
print(model_par.summary())

## Model with interaction (different slopes)

In [ ]:
model_slopes = smf.ols("vacancies ~ size * n_firms", data = df).fit()
print(model_slopes.summary())

## Slopes by group
b = model_slopes.params
print(f"Slope for Large: {b['n_firms']:.6f}")
print(f"Slope for Medium: {b['n_firms'] + b['size[T.Medium]:n_firms']:.6f}")
print(f"Slope for Small: {b['n_firms'] + b['size[T.Small]:n_firms']:.6f}")

## Visualising parallel vs. crossing lines

In [ ]:
df["fitted_par"] = model_par.fittedvalues
df["fitted_int"] = model_slopes.fittedvalues

plot_data = df.groupby(["n_firms", "size"])[["fitted_par", "fitted_int"]].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, grp in plot_data.groupby("size"):
    axes[0].plot(grp["n_firms"], grp["fitted_par"], label=name)
    axes[1].plot(grp["n_firms"], grp["fitted_int"], label=name)
axes[0].set_title("Without interaction (parallel lines)")
axes[1].set_title("With interaction (different slopes)")
for ax in axes:
    ax.set_xlabel("Number of firms in sector (thousands)")
    ax.set_ylabel("Fitted vacancies")
    ax.legend()
plt.tight_layout()
plt.show()

## Grand-mean centering

In [ ]:
df["n_firms_c"] = df["n_firms"] - df["n_firms"].mean()

model_cent = smf.ols("vacancies ~ size * n_firms_c", data = df).fit()
print(model_cent.summary())

## What changes and what stays the same?

In [ ]:
## R-squared: identical
print(f"R² (uncentered): {model_slopes.rsquared:.6f}")
print(f"R² (centered):   {model_cent.rsquared:.6f}")

## Interaction coefficients: identical
print(f"\nInteraction coefs (uncentered):")
print(model_slopes.params.filter(like=":"))
print(f"Interaction coefs (centered):")
print(model_cent.params.filter(like=":"))

## Main-effect coefficients: different
print(f"\nMain effects (uncentered):")
print(model_slopes.params.iloc[:3])
print(f"Main effects (centered):")
print(model_cent.params.iloc[:3])

## Standardisation (z-score)

In [ ]:
df["n_firms_s"] = (df["n_firms"] - df["n_firms"].mean()) / df["n_firms"].std()

model_std = smf.ols("vacancies ~ size * n_firms_s", data = df).fit()
print(model_std.summary())

## Implementation

In [ ]:
## Gelman scaling
df["n_firms_g"] = (df["n_firms"] - df["n_firms"].mean()) / (2 * df["n_firms"].std())

## Center binary by its mean
df["public_num"] = (df["public"] == "1").astype(float)
df["public_c"] = df["public_num"] - df["public_num"].mean()

model_gelman = smf.ols("vacancies ~ size + n_firms_g + public_c", data=df).fit()
print(model_gelman.summary())

## Comparison: coefficient magnitudes

In [ ]:
model_orig = smf.ols("vacancies ~ size + n_firms + public_num", data=df).fit()
model_1sd = smf.ols("vacancies ~ size + n_firms_s + public_num", data=df).fit()
model_2sd = smf.ols("vacancies ~ size + n_firms_g + public_c", data=df).fit()

comparison = pd.DataFrame({
    "original": model_orig.params,
    "z_score_1sd": model_1sd.params,
    "gelman_2sd": model_2sd.params
})
print(comparison.round(4))